In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("event-logs.csv")


# Quick look at the shape of the data

-50,240 rows  
-3% of the data in the 'Entity Details' column is null (i.e. missing)  
-All column data types are string objects (text)  

In [110]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50240 entries, 0 to 50239
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   Timestamp       50240 non-null  datetime64[ns, UTC]
 1   User            50240 non-null  str                
 2   Event Type      50240 non-null  str                
 3   Entity Type     50240 non-null  str                
 4   Entity Details  50240 non-null  str                
 5   Event Id        50240 non-null  str                
 6   Event Action    50240 non-null  str                
 7   DateString      50240 non-null  str                
 8   Year            50240 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(1), str(7)
memory usage: 3.4 MB


In [112]:
df.head(2)

,Timestamp,User,Event Type,Entity Type,Entity Details,Event Id,Event Action,DateString,Year
0,2026-04-25 03:35:06.211000+00:00,Investigations,read,stream,Interaction: View Live Stream\nCamera Name: C#...,e6f0f896-7937-505c-bd1a-99cdec24f680,Interaction: View Live Stream,04/25/2026,2026
1,2026-04-25 03:35:05.934000+00:00,Investigations,read,stream,Interaction: View Live Stream\nCamera Name: C#...,26661c87-f7c8-540b-8774-7bad7909bf19,Interaction: View Live Stream,04/25/2026,2026


In [ ]:
# Fill in a bunch of empty data in the 'Entity Details' column
df = df.fillna("", inplace = True)


# Check 'Entity Details' column

This column contains descriptive details about the entity (seems to be mostly the cameras) that receives the event/action

In [113]:
df['Entity Details'].value_counts().to_frame('count').reset_index().head(20).iloc[6]['Entity Details']

'Interaction: View Live Stream\nCamera Name: C#004 Planet Fitness\n'

#### Let's see how many cameras are connected to the system...

In [114]:
# Extract the name of the unique cameras from the column
uniquecameras = []

for i in range(len(df)):

    # some values are floats, ignore these
    if type(df['Entity Details'][i]) != float:

        # split the list object on new line
        x = df['Entity Details'][i].split("\n")
        
        for j in x:
            
            # search the string for the substring 'Camera Name' and add to list 
            # if it's not there to get the unique camera names
            if 'Camera Name' in j and j not in uniquecameras:
                uniquecameras.append(j)

            else:
                pass

#### The city of Coon Rapids has an MOU with Anoka County for [12 Flock ALPRs](https://public.destinyhosted.com/agenda_publish.cfm?dsp=agm&seq=10032&rev=0&id=26667&form_type=AG_MEMO&beg_meetmth=1&beg_meetyr=2015&end_meetmth=7&end_meetyr=2026&mt=ALL&sstr=Flock&dept=ALL&hartkeywords=&sortby=f.form_num,%20f.rev_num&fp=ADVSRCH&StartRow=1)
##### This is in addition to the [6 Flock ALPRs in Riverdale](https://public.destinyhosted.com/agenda_publish.cfm?dsp=agm&seq=9617&rev=0&id=26667&form_type=AG_MEMO&beg_meetmth=1&beg_meetyr=2015&end_meetmth=7&end_meetyr=2026&mt=ALL&sstr=Flock&dept=ALL&hartkeywords=&sortby=f.form_num,%20f.rev_num&fp=ADVSRCH&StartRow=1)
##### However, it is clear in the data below that the city has connected many of their already-owned cameras into the Flock system.

In [115]:
len(uniquecameras)

99

In [116]:
uniquecameras

['Camera Name: C#015 Lions Park Restrooms',
 'Camera Name: C#013 Lions Park Basketball',
 'Camera Name: C#004 Planet Fitness',
 'Camera Name: C#006 Ulta Beauty @ River Rapids Dr - PTZ',
 "Camera Name: C#008 Dick's Sporting Goods",
 'Camera Name: C#001 Round Lake/Riverdale Blvd',
 'Camera Name: C#002 Riverdale Blvd @ Main St - PTZ',
 'Camera Name: C#007 Famous Footwear',
 'Camera Name: C#009 Buffalo Wild Wings @ River Rapids Dr - PTZ',
 'Camera Name: C#04 Crooked Lake Shelter 2',
 'Camera Name: C#03 Crooked Lake Restrooms',
 'Camera Name: C#05 Crooked Lake Parking lot',
 'Camera Name: C#01 Crooked Lake Beach',
 'Camera Name: C#02 Crooked Lake Fishing Piers',
 'Camera Name: C#014 Lions Park Shelter 2',
 'Camera Name: C#016 Lions Park Playground',
 'Camera Name: C#006 Lions Park South Lot',
 'Camera Name: C#012 Schneiderman’s Side Lot',
 'Camera Name: C#003 Riverdale Village Playground @ River Rapids Dr - PTZ',
 'Camera Name: C#010 Noodles & Company',
 'Camera Name: C#011 Schneiderman’s',

# A little data prep...

In [118]:
# Create 'Event Action' column, which contains the first string element
# from the string (split on new line) in the 'Entity Details' column.
# This is the 'interaction' type
df['Event Action'] = [x.split("\n")[0] for x in df['Entity Details']]

In [119]:
# Examine the event types (the kinds of actions taken in the system)
df['Event Type'].value_counts()

Event Type
update    35077
read      14860
create      253
delete       50
Name: count, dtype: int64

In [120]:
# Create DataFrames for each Event Type
dfcreate = df[df['Event Type'] == 'create'].reset_index(drop = True)
dfdelete = df[df['Event Type'] == 'delete'].reset_index(drop = True)
dfread = df[df['Event Type'] == 'read'].reset_index(drop = True)
dfupdate = df[df['Event Type'] == 'update'].reset_index(drop = True)

In [121]:
# All Entity Types
df['Entity Type'].value_counts()

Entity Type
stream                     47822
search                      1478
role                         370
user                         334
Custom Hotlist Entry         109
networkShare                  36
customHotlist                 30
location                      14
organization                  13
networkShareSettings           9
vehicleDescriptionAlert        8
shareRequest                   7
accountContact                 7
integration                    3
Name: count, dtype: int64

# Exploratory Data Analaysis (EDA)
#### Let's dig in

Quick look at integrations. What's being integrated into the system?

In [122]:
df[df['Entity Type'] == 'integration']

,Timestamp,User,Event Type,Entity Type,Entity Details,Event Id,Event Action,DateString,Year
11214,2026-03-12 16:10:40.002000+00:00,Cindy Hintze,create,integration,Message: Creating an Integration\nIntegration ...,fbd125bc-a6a1-5893-9f04-82ba371df272,Message: Creating an Integration,03/12/2026,2026
44913,2025-07-15 14:32:00.974000+00:00,Adam Knauer,create,integration,Message: Creating an Integration\nIntegration ...,23490bc5-cfd6-5bb7-9466-e884c6a8a024,Message: Creating an Integration,07/15/2025,2025
44955,2025-07-15 12:20:50.155000+00:00,Adam Knauer,create,integration,Message: Creating an Integration\nIntegration ...,84dc4e97-95b7-59bf-a78b-00bcf06ae238,Message: Creating an Integration,07/15/2025,2025


In [123]:
# Axon evidence system??
df[df['Entity Type'] == 'integration']['Entity Details'][44913].split("\n")

['Message: Creating an Integration',
 'Integration Name: Axon Evidence',
 'Catalog Entry ID: b5bc1746-cb0e-4b73-a5c3-afc8345f4de2',
 '']

In [124]:
# This is likely Microsoft SSO for users to sign into the system using their ciy Microsoft acount
df[df['Entity Type'] == 'integration']['Entity Details'][44955].split("\n")

['Message: Creating an Integration',
 'Integration Name: Microsoft Entra ID',
 'Catalog Entry ID: MicrosoftEntraID',
 '']

In [125]:
# An API integration with a big technology company. 
# That's...concerning.
df[df['Entity Type'] == 'integration']['Entity Details'][11214].split("\n")

['Message: Creating an Integration',
 'Integration Name: V3 API (Tyler Technologies)',
 'Catalog Entry ID: f8c8d615-959d-4906-9eec-b4e6f6deb508',
 '']

### Ok. Let's start small: What kinds of things are 'created'?

Seems to be mainly the creation of users, but also hotlist entries

In [ ]:
dfcreate['Entity Type'].value_counts()

Entity Type
user                       101
Custom Hotlist Entry        78
networkShare                34
customHotlist               13
role                         9
shareRequest                 7
accountContact               4
integration                  3
vehicleDescriptionAlert      3
organization                 1
Name: count, dtype: int64

In [ ]:
# Make a networkShare DataFrame to see who has been given access to the Coon Rapids database
dfns = dfcreate[dfcreate['Entity Type'] == 'networkShare'].reset_index(drop = True)

In [ ]:
# See what the data looks like...
dfns['Entity Details'][4].split("\n")

In [ ]:
# Network shares show the organizations with whom Coon Rapids is sharing access.
# Make a list of all orgs who have access to CR cameras.
orgs = []

for i in range(len(dfns)):

    # grab the third item in the split list, remove text so as to keep the org name only
    o = dfns['Entity Details'][i].split("\n")[2].replace("Receiving Organization: ", "")

    if o not in orgs:
        orgs.append(o)


In [ ]:
# The entities that have varying levels of access to the Coon Rapids Flock system
orgs

# Read

In [ ]:
dfread['Event Action'].value_counts()

In [ ]:
# 'Read' events are of two types: watching a camera's stream, and searching for something
dfread['Entity Type'].value_counts()

Let's make a searches DataFrame and see who is searching for what

In [ ]:
dfsearches = dfread[dfread['Entity Type'] == 'search'].reset_index(drop = True)

In [ ]:
# No search data is preserved in the log, 
# so we don't know what they're typing when they search
dfsearches['Entity Details'].value_counts()

In [ ]:
# But who has done the searching?
dfsearches['User'].value_counts()

In [ ]:
# Searches over time
dfsearches['Timestamp'] = pd.to_datetime(dfsearches['Timestamp'])

# Add DateString column and month and year columns for groupby counting
dfsearches['DateString'] = [x.strftime("%m/%d/%Y") for x in dfsearches['Timestamp']]
dfsearches['Year'] = [x.year for x in dfsearches['Timestamp']]
dfsearches['Month'] = [x.month for x in dfsearches['Timestamp']]
dfsearches['Day'] = [x.day for x in dfsearches['Timestamp']]

In [ ]:
dfSearch2 = dfsearches.groupby(['DateString'])['Event Action'].count().to_frame('Search Count').reset_index()

In [ ]:
dfSearch2['SortDate'] = pd.to_datetime(dfSearch2['DateString'])

In [ ]:
dfSearch2Sort = dfSearch2.sort_values(['SortDate']).reset_index(drop = True)

In [ ]:
len(dfsearches)

In [ ]:
dfSearch2Sort.plot(kind = 'line', x = 'DateString', y = 'Search Count', rot = 45);

In [ ]:
# The search spike in early February correlates with heightened Metro Surge activity in MN.
# Let's check out who searched.
dfsearchFeb = dfsearches[dfsearches['Month'] == 2].reset_index(drop = True)

In [ ]:
dfsearchFeb10 = dfsearchFeb[(dfsearchFeb['Day'] == 10)].reset_index(drop = True)

In [ ]:
dfsearchFeb10['User'].value_counts()

# Who's viewing which livestreams - 'read'

In [ ]:
# Alright, let's dig in a little more.
dflive = df[df['Event Action'] == 'Interaction: View Live Stream'].reset_index(drop = True)

In [ ]:
dflive.head()

In [ ]:
# Get the second line of text from the 'Entity Details' column. 
# This seems to be the camera name
dflive['Camera'] = [x.split("\n")[1] for x in dflive['Entity Details']]

In [ ]:
# They keep an eye on Lion's Park my goodness
dflive.Camera.value_counts()

# 'Update' event types
#### This is the most prominent kind of action in the system. 70% of all activity is 'update'

In [ ]:
# Quick look
dfupdate.head()

In [ ]:
# Count of entity types in the 'update' DataFrame
# Primarily viewing livestreams
dfupdate['Entity Type'].value_counts()

In [ ]:
# NETWORKSHARES, VEHICLEDESCRIPTIONALERTS, ACCOUNTCONTACTS, CUSTOMHOTLIST, CUSTOM HOTLIST ENTRY,
# NETWORK SHARE SETTINGS, ORGANIZATIONS, LOCATIONS, USERS, ROLES
dfUpdNS = dfupdate[dfupdate['Entity Type'] == 'networkShare'].reset_index(drop = True)
dfUpdVDA = dfupdate[dfupdate['Entity Type'] == 'vehicleDescriptionAlert'].reset_index(drop = True)
dfUpdAC = dfupdate[dfupdate['Entity Type'] == 'accountContact'].reset_index(drop = True)
dfUpdCH = dfupdate[dfupdate['Entity Type'] == 'customHotlist'].reset_index(drop = True)
dfUpdCHE = dfupdate[dfupdate['Entity Type'] == 'Custom Hotlist Entry'].reset_index(drop = True)
dfUpdNet = dfupdate[dfupdate['Entity Type'] == 'networkShareSettings'].reset_index(drop = True)
dfUpdOrg = dfupdate[dfupdate['Entity Type'] == 'organization'].reset_index(drop = True)
dfUpdLoc = dfupdate[dfupdate['Entity Type'] == 'location'].reset_index(drop = True)
dfUpdUser = dfupdate[dfupdate['Entity Type'] == 'user'].reset_index(drop = True)
dfUpdRole = dfupdate[dfupdate['Entity Type'] == 'role'].reset_index(drop = True)

In [ ]:
dfUpdOrg.head(25)

# WOAH Immigration Enforcement? Isn't that supposed to be a Fed only activity?

In [ ]:
dfUpdOrg.iloc[2]['Entity Details'].split("\n")

## Ok, what on earth is this? It appears multiple 'networks' (whatever those are) can search in Coon Rapids Flock system for immigration enforcement and reproductive health???? What?


### Livestreams 'update' entity type

In [ ]:
dfupStream = dfupdate[dfupdate['Entity Type'] == 'stream'].reset_index(drop = True)

In [ ]:
dfupStream['Camera'] = [x.split("\n")[1] for x in dfupStream['Entity Details']]

In [ ]:
dfupStream['Camera'].value_counts()

In [ ]:
dfupStream2 = dfupStream[dfupStream['Event Action'] == 'Interaction: View Live Stream'].reset_index(drop = True)

In [ ]:
dfupStream2.Camera.value_counts().to_frame('count').head(20)

# Overall live stream views over time

In [ ]:
dflivestream = df[df['Event Action'] == 'Interaction: View Live Stream'].reset_index(drop = True)

In [ ]:
dflivestream['Timestamp'] = pd.to_datetime(dflivestream['Timestamp'])

In [ ]:
# Add DateString column and month and year columns for groupby counting
dflivestream['DateString'] = [x.strftime("%m/%d/%Y") for x in dflivestream['Timestamp']]
dflivestream['DateStringAMPM'] = [x.strftime("%m/%d/%Y %I:%M %p") for x in dflivestream['Timestamp']]
dflivestream['Year'] = [x.year for x in dflivestream['Timestamp']]
dflivestream['Month'] = [x.month for x in dflivestream['Timestamp']]
dflivestream['Day'] = [x.day for x in dflivestream['Timestamp']]

In [ ]:
dfLS2 = dflivestream.groupby(['Year', 'Month', 'Day'])['Event Action'].count().to_frame('Livestream Views').reset_index()

In [ ]:
dfLS3 = dflivestream.groupby(['DateString'])['Event Action'].count().to_frame('Livestream Views').reset_index()

In [ ]:
dfLS3['SortDate'] = pd.to_datetime(dfLS3['DateString'])

In [ ]:
dfLS3sort = dfLS3.sort_values(['SortDate']).reset_index(drop = True)

In [ ]:
dfLS3sort.plot(kind = 'line', x = 'DateString', y = 'Livestream Views', rot = 45);

In [ ]:
df2 = df

In [ ]:
df2['Timestamp'] = pd.to_datetime(df2['Timestamp'])

In [ ]:
df2['DateString'] = [x.strftime("%m/%d/%Y") for x in df2['Timestamp']]
df2['Year'] = [x.year for x in df2['Timestamp']]

In [ ]:
dfYear = df2.groupby(['Year'])['Year'].count().to_frame('count').reset_index()

In [ ]:
dfYear.plot(x = 'Year', y = 'count', kind = 'bar', title = 'Events by Year')